# 12 Check Data Dataset 2

This notebook audits Dataset-Bacteria and writes the clean image manifest used by the later dataset 2 notebooks.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import json

import pandas as pd

from IPython.display import display
from PIL import Image

# This helper keeps the notebook easy to run from the repo root or from notebooks/.
def find_repo_root(start_path: Path) -> Path:
    if (start_path / "raw_data").exists():
        return start_path
    if start_path.name == "notebooks" and (start_path.parent / "raw_data").exists():
        return start_path.parent
    raise FileNotFoundError("Could not find the FYP2 repo root.")

REPO_ROOT = find_repo_root(Path.cwd())
DATASET_DIR = REPO_ROOT / "Dataset-Bacteria"
MANIFESTS_DIR = REPO_ROOT / "manifests"
RESULTS_DIR = REPO_ROOT / "results"
NOTEBOOK_TAG = "12_check_data_dataset2"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / NOTEBOOK_TAG
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_MAPPING = {
    "Enterococcus faecalis": {"species_name": "Enterococcus.faecalis", "binary_group": "cocci", "binary_label": 0},
    "Enterococcus faecium": {"species_name": "Enterococcus.faecium", "binary_group": "cocci", "binary_label": 0},
    "Staphylococcus aureus": {"species_name": "Staphylococcus.aureus", "binary_group": "cocci", "binary_label": 0},
    "Clostridium perfringens": {"species_name": "Clostridium.perfringens", "binary_group": "bacilli", "binary_label": 1},
    "Escherichia coli": {"species_name": "Escherichia.coli", "binary_group": "bacilli", "binary_label": 1},
    "Listeria monocytogenes": {"species_name": "Listeria.monocytogenes", "binary_group": "bacilli", "binary_label": 1},
    "Pseudomonas aeruginosa": {"species_name": "Pseudomonas.aeruginosa", "binary_group": "bacilli", "binary_label": 1},
}

EXPECTED_IMAGE_COUNTS = {
    "Clostridium.perfringens": 62,
    "Enterococcus.faecalis": 60,
    "Enterococcus.faecium": 56,
    "Escherichia.coli": 59,
    "Listeria.monocytogenes": 63,
    "Pseudomonas.aeruginosa": 63,
    "Staphylococcus.aureus": 64,
}
EXPECTED_IMAGE_COUNT = 427
EXPECTED_SPECIES_COUNT = 7
EXPECTED_DUPLICATE_COUNT = 0

print(f"Repo root: {REPO_ROOT}")
print(f"Notebook tag: {NOTEBOOK_TAG}")
print(f"Dataset dir: {DATASET_DIR}")

Repo root: C:\Users\FYP2610\Downloads\FYP2
Notebook tag: 12_check_data_dataset2
Dataset dir: C:\Users\FYP2610\Downloads\FYP2\Dataset-Bacteria


In [2]:
# This helper writes a CSV file with a stable header.
def write_csv_rows(csv_path: Path, rows, fieldnames):
    if not rows:
        raise ValueError(f"No rows were provided for {csv_path.name}.")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# This helper writes one small JSON file with clean formatting.
def write_json(json_path: Path, data):
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

# This helper stores paths in repo-relative form so the outputs stay portable.
def as_repo_relative(repo_root: Path, path_value: Path) -> str:
    return path_value.relative_to(repo_root).as_posix()

In [3]:
# We audit every PNG file and keep a stable RGBA hash for duplicate checks.
def compute_rgba_sha256(image_path: Path) -> str:
    with Image.open(image_path) as image_handle:
        rgba_image = image_handle.convert("RGBA")
        return hashlib.sha256(rgba_image.tobytes()).hexdigest()

image_rows = []
hash_to_paths = defaultdict(list)

for species_dir in sorted(path for path in DATASET_DIR.iterdir() if path.is_dir()):
    species_name_raw = species_dir.name
    if species_name_raw not in SPECIES_MAPPING:
        raise ValueError(f"Unexpected dataset 2 species folder: {species_name_raw}")

    mapping = SPECIES_MAPPING[species_name_raw]
    for image_path in sorted(species_dir.glob("*.png")):
        with Image.open(image_path) as image_handle:
            rgba_image = image_handle.convert("RGBA")
            width, height = rgba_image.size
            alpha_min, alpha_max = rgba_image.getchannel("A").getextrema()
            mode = image_handle.mode

        image_id = f"{mapping['species_name']}_{image_path.stem}"
        rgba_sha256 = compute_rgba_sha256(image_path)
        hash_to_paths[rgba_sha256].append(image_path)
        image_rows.append({
            "image_id": image_id,
            "species_name_raw": species_name_raw,
            "species_name": mapping["species_name"],
            "binary_group": mapping["binary_group"],
            "binary_label": mapping["binary_label"],
            "width": width,
            "height": height,
            "mode": mode,
            "has_alpha": int(mode.endswith("A")),
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "rgba_sha256": rgba_sha256,
            "file_path": as_repo_relative(REPO_ROOT, image_path),
        })

duplicate_rows = []
exact_duplicate_count = 0
for row in image_rows:
    duplicate_group_size = len(hash_to_paths[row["rgba_sha256"]])
    is_duplicate = duplicate_group_size > 1
    if is_duplicate:
        exact_duplicate_count += 1
    duplicate_rows.append({
        "image_id": row["image_id"],
        "species_name": row["species_name"],
        "duplicate_group_size": duplicate_group_size,
        "is_exact_duplicate": int(is_duplicate),
        "file_path": row["file_path"],
    })

if len(image_rows) != EXPECTED_IMAGE_COUNT:
    raise AssertionError(f"Expected {EXPECTED_IMAGE_COUNT} dataset 2 images, but found {len(image_rows)}.")

species_counts = Counter(row["species_name"] for row in image_rows)
size_counts = Counter((row["width"], row["height"]) for row in image_rows)
binary_counts = Counter(row["binary_group"] for row in image_rows)

if dict(species_counts) != EXPECTED_IMAGE_COUNTS:
    raise AssertionError(f"Species counts do not match the expected dataset 2 set: {species_counts}")
if len(species_counts) != EXPECTED_SPECIES_COUNT:
    raise AssertionError("Unexpected number of dataset 2 species.")
if exact_duplicate_count != EXPECTED_DUPLICATE_COUNT:
    raise AssertionError(f"Expected {EXPECTED_DUPLICATE_COUNT} exact duplicates, but found {exact_duplicate_count}.")

image_manifest_path = MANIFESTS_DIR / "dataset2_image_manifest.csv"
write_csv_rows(image_manifest_path, image_rows, list(image_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "duplicate_audit.csv", duplicate_rows, list(duplicate_rows[0].keys()))

species_rows = [{"species_name": key, "image_count": species_counts[key]} for key in sorted(species_counts)]
binary_rows = [{"binary_group": key, "image_count": binary_counts[key]} for key in sorted(binary_counts)]
size_rows = [{"width": key[0], "height": key[1], "image_count": value} for key, value in sorted(size_counts.items())]

write_csv_rows(NOTEBOOK_RESULTS_DIR / "species_counts.csv", species_rows, list(species_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "binary_counts.csv", binary_rows, list(binary_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "image_size_counts.csv", size_rows, list(size_rows[0].keys()))
write_json(
    NOTEBOOK_RESULTS_DIR / "summary.json",
    {
        "notebook_tag": NOTEBOOK_TAG,
        "image_count": len(image_rows),
        "species_count": len(species_counts),
        "cocci_image_count": binary_counts["cocci"],
        "bacilli_image_count": binary_counts["bacilli"],
        "exact_duplicate_count": exact_duplicate_count,
        "unique_image_size_count": len(size_counts),
        "image_manifest_path": as_repo_relative(REPO_ROOT, image_manifest_path),
    },
)

display(pd.DataFrame(image_rows[:5]))
display(pd.DataFrame(species_rows))
display(pd.DataFrame(binary_rows))
display(pd.DataFrame(size_rows))
print(f"Saved dataset 2 image manifest to: {image_manifest_path}")

,image_id,species_name_raw,species_name,binary_group,binary_label,width,height,mode,has_alpha,alpha_min,alpha_max,rgba_sha256,file_path
0,Clostridium.perfringens_01,Clostridium perfringens,Clostridium.perfringens,bacilli,1,1225,1225,RGBA,1,0,255,e2a6dafd04a0747f90c226822a3240bf4ebb87a62ad8ab...,Dataset-Bacteria/Clostridium perfringens/01.png
1,Clostridium.perfringens_02,Clostridium perfringens,Clostridium.perfringens,bacilli,1,1225,1225,RGBA,1,0,255,fc3c7e73c2bcbae38b030b6276bcd964ae88d170e54c9d...,Dataset-Bacteria/Clostridium perfringens/02.png
2,Clostridium.perfringens_03,Clostridium perfringens,Clostridium.perfringens,bacilli,1,1225,1225,RGBA,1,0,255,9c3d9e1d80542938a2ef625228f1625478bb6f2b374cec...,Dataset-Bacteria/Clostridium perfringens/03.png
3,Clostridium.perfringens_04,Clostridium perfringens,Clostridium.perfringens,bacilli,1,1225,1225,RGBA,1,0,255,bcef67b240e1ec3aff8801c692fdfe39b71aab7f69d2d5...,Dataset-Bacteria/Clostridium perfringens/04.png
4,Clostridium.perfringens_05,Clostridium perfringens,Clostridium.perfringens,bacilli,1,1225,1225,RGBA,1,0,255,ec6b5a663d013586968ae1ea39b1e0d4d566484e6c11a7...,Dataset-Bacteria/Clostridium perfringens/05.png


,species_name,image_count
0,Clostridium.perfringens,62
1,Enterococcus.faecalis,60
2,Enterococcus.faecium,56
3,Escherichia.coli,59
4,Listeria.monocytogenes,63
5,Pseudomonas.aeruginosa,63
6,Staphylococcus.aureus,64


,binary_group,image_count
0,bacilli,247
1,cocci,180


,width,height,image_count
0,832,832,2
1,864,864,3
2,911,911,4
3,916,916,5
4,927,927,3
5,938,938,3
6,942,942,4
7,946,946,3
8,951,951,3
9,962,962,3


Saved dataset 2 image manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\dataset2_image_manifest.csv
